In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from tqdm.auto import tqdm


# ============================================================
# ============   USER-DEFINED PARAMETERS   ===================
# ============================================================
#Replace file path with actual path

INPUT_DIR        =  "data/database"                               # where xlsx files are
OUTPUT_DIR       =  "data/output"                         # where outputs go
TRAIN_FILE       = os.path.join(INPUT_DIR, "input.xlsx")
VALID_FILE       = os.path.join(INPUT_DIR, "38-experimental-validation.xlsx")

ESM_MODEL_NAME   = "facebook/esm2_t12_35M_UR50D"   # small & fast; try t30_150M for higher capacity
LABELS_TO_USE    = ["aggregate", "LLPS", "soluble"]  # drop 'unknown'
INCLUDE_LIPID    = True                              # add lipidation as token prefix
MAX_LEN          = 32
BATCH_SIZE       = 32
EPOCHS           = 10
LR               = 2e-5
WEIGHT_DECAY     = 0.01
SEED             = 42
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"
TEST_SIZE        = 0.2

os.makedirs(OUTPUT_DIR, exist_ok=True)
torch.manual_seed(SEED); np.random.seed(SEED)
print("Device:", DEVICE)

In [ ]:
def load_clean(path):
    df = pd.read_excel(path)
    df.columns = [c.strip().lower() for c in df.columns]
    # standardise expected columns
    needed = {"sequence", "category"}
    if not needed.issubset(df.columns):
        raise ValueError(f"{path} must contain columns: {needed}")
    if "lipidation" not in df.columns:
        df["lipidation"] = "none"
    df["sequence"]   = df["sequence"].astype(str).str.strip().str.upper()
    df["category"]   = df["category"].astype(str).str.strip()
    df["lipidation"] = df["lipidation"].astype(str).str.strip().str.lower().fillna("none")
    df = df[df["category"].isin(LABELS_TO_USE)].reset_index(drop=True)
    df = df.drop_duplicates(subset=["sequence", "lipidation"]).reset_index(drop=True)
    return df

train_df = load_clean(TRAIN_FILE)
valid_df = load_clean(VALID_FILE)

print("Training data class distribution:")
print(train_df["category"].value_counts())
print("\nValidation data class distribution:")
print(valid_df["category"].value_counts())

label2id = {l: i for i, l in enumerate(LABELS_TO_USE)}
id2label = {i: l for l, i in label2id.items()}
print("\nLabel map:", label2id)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(ESM_MODEL_NAME)

class PeptideDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=MAX_LEN, include_lipid=INCLUDE_LIPID):
        self.df = df.reset_index(drop=True)
        self.tok = tokenizer
        self.max_len = max_len
        self.include_lipid = include_lipid

    def __len__(self):
        return len(self.df)

    def _format_seq(self, row):
        seq = row["sequence"]
        if self.include_lipid and row["lipidation"] not in ("none", "nan", ""):
            # prefix as a non-canonical tag — ESM tokenizer will handle unknowns gracefully
            return f"{row['lipidation'][:3].upper()}-{seq}"
        return seq

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = self._format_seq(row)
        enc = self.tok(text, truncation=True, padding="max_length",
                       max_length=self.max_len, return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(label2id[row["category"]], dtype=torch.long)
        return item

# Train/test split (stratified)
tr_df, te_df = train_test_split(
    train_df, test_size=TEST_SIZE, random_state=SEED,
    stratify=train_df["category"]
)
train_ds = PeptideDataset(tr_df, tokenizer)
test_ds  = PeptideDataset(te_df, tokenizer)
ext_ds   = PeptideDataset(valid_df, tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)
ext_loader   = DataLoader(ext_ds,   batch_size=BATCH_SIZE)

print(f"Train: {len(train_ds)} | Internal-test: {len(test_ds)} | External-validation: {len(ext_ds)}")

In [ ]:
class ESM2Classifier(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.1):
        super().__init__()
        self.esm = AutoModel.from_pretrained(model_name)
        hidden = self.esm.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, num_labels)

    def forward(self, input_ids, attention_mask, labels=None):
        out = self.esm(input_ids=input_ids, attention_mask=attention_mask)
        # mean pool over valid tokens
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1)
        logits = self.classifier(self.dropout(pooled))
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
        return loss, logits

model = ESM2Classifier(ESM_MODEL_NAME, num_labels=len(LABELS_TO_USE)).to(DEVICE)

# class weights for imbalance
class_counts = tr_df["category"].value_counts().reindex(LABELS_TO_USE).values
class_weights = torch.tensor(class_counts.sum() / (len(class_counts) * class_counts),
                             dtype=torch.float, device=DEVICE)
print("Class weights:", dict(zip(LABELS_TO_USE, class_weights.tolist())))

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

history = {"train_loss": [], "test_acc": []}

def evaluate(loader):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            _, logits = model(batch["input_ids"], batch["attention_mask"])
            all_logits.append(logits.cpu().numpy())
            all_labels.append(batch["labels"].cpu().numpy())
    logits = np.concatenate(all_logits)
    labels = np.concatenate(all_labels)
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = probs.argmax(axis=1)
    return labels, preds, probs

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for batch in pbar:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        optimizer.zero_grad()
        _, logits = model(batch["input_ids"], batch["attention_mask"])
        loss = loss_fn(logits, batch["labels"])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        running += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    avg = running / len(train_loader)
    y_true, y_pred, _ = evaluate(test_loader)
    acc = accuracy_score(y_true, y_pred)
    history["train_loss"].append(avg); history["test_acc"].append(acc)
    print(f"Epoch {epoch}: train_loss={avg:.4f} | test_acc={acc:.4f}")

# Save model
model_path = os.path.join(OUTPUT_DIR, "esm2_classifier.pt")
torch.save({"state_dict": model.state_dict(),
            "label2id": label2id,
            "id2label": id2label,
            "model_name": ESM_MODEL_NAME,
            "max_len": MAX_LEN,
            "include_lipid": INCLUDE_LIPID}, model_path)
print(f"Model saved to {model_path}")

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(range(1, EPOCHS + 1), history["train_loss"], "o-")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Train loss"); axes[0].set_title("Training loss")
axes[1].plot(range(1, EPOCHS + 1), history["test_acc"], "o-", color="green")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Test accuracy"); axes[1].set_title("Internal test accuracy")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=200)
plt.show()

In [13]:
def full_evaluation(loader, tag, df_for_export=None):
    y_true, y_pred, probs = evaluate(loader)

    # ---- Metrics ----
    acc = accuracy_score(y_true, y_pred)
    print(f"\n=== {tag} ===")
    print(f"Accuracy: {acc:.4f}")
    print("\nClassification report:")
    print(classification_report(y_true, y_pred,
                                target_names=LABELS_TO_USE, digits=4, zero_division=0))

    # ---- Confusion matrix ----
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(LABELS_TO_USE))))
    plt.figure(figsize=(5.5, 4.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=LABELS_TO_USE, yticklabels=LABELS_TO_USE)
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"Confusion matrix — {tag}")
    plt.tight_layout()
    cm_path = os.path.join(OUTPUT_DIR, f"confusion_matrix_{tag}.png")
    plt.savefig(cm_path, dpi=200); plt.show()

    # ---- ROC / AUROC (one-vs-rest) ----
    y_bin = label_binarize(y_true, classes=list(range(len(LABELS_TO_USE))))
    auroc_per_class = {}
    plt.figure(figsize=(6, 5))
    for i, lab in enumerate(LABELS_TO_USE):
        if y_bin[:, i].sum() == 0:
            print(f"  [skip ROC for {lab}: no positives in {tag}]")
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
        a = auc(fpr, tpr)
        auroc_per_class[lab] = a
        plt.plot(fpr, tpr, label=f"{lab} (AUC={a:.3f})")
    plt.plot([0, 1], [0, 1], "k--", lw=1)
    plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
    plt.title(f"ROC — {tag}")
    plt.legend(loc="lower right"); plt.tight_layout()
    roc_path = os.path.join(OUTPUT_DIR, f"roc_{tag}.png")
    plt.savefig(roc_path, dpi=200); plt.show()

    # macro AUROC (only over classes present)
    try:
        present = [i for i, lab in enumerate(LABELS_TO_USE) if y_bin[:, i].sum() > 0]
        if len(present) >= 2:
            macro = roc_auc_score(y_bin[:, present], probs[:, present],
                                  average="macro", multi_class="ovr")
            print(f"Macro AUROC ({tag}): {macro:.4f}")
        else:
            macro = float("nan")
    except Exception as e:
        macro = float("nan")
        print("Macro AUROC could not be computed:", e)

    # ---- Save predictions to CSV ----
    if df_for_export is not None:
        out = df_for_export.copy().reset_index(drop=True)
        out["true_label"] = [id2label[i] for i in y_true]
        out["pred_label"] = [id2label[i] for i in y_pred]
        for i, lab in enumerate(LABELS_TO_USE):
            out[f"prob_{lab}"] = probs[:, i]
        out["correct"] = (out["true_label"] == out["pred_label"])
        csv_path = os.path.join(OUTPUT_DIR, f"predictions_{tag}.csv")
        out.to_csv(csv_path, index=False)
        print(f"Predictions saved to {csv_path}")

    return {"accuracy": acc, "auroc_per_class": auroc_per_class,
            "macro_auroc": macro, "confusion_matrix": cm.tolist()}

In [ ]:
internal_metrics = full_evaluation(test_loader, "internal_test", df_for_export=te_df)

In [ ]:
external_metrics = full_evaluation(ext_loader, "external_validation", df_for_export=valid_df)

In [ ]:
summary = {
    "model": ESM_MODEL_NAME,
    "labels": LABELS_TO_USE,
    "include_lipidation": INCLUDE_LIPID,
    "max_len": MAX_LEN,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "internal_test": internal_metrics,
    "external_validation": external_metrics,
}
with open(os.path.join(OUTPUT_DIR, "summary_metrics.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("\nAll outputs saved in:", OUTPUT_DIR)
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", f)

In [ ]:
def predict_sequences(sequences, lipidations=None, ckpt_path=model_path):
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    tok = AutoTokenizer.from_pretrained(ckpt["model_name"])
    mdl = ESM2Classifier(ckpt["model_name"], num_labels=len(ckpt["label2id"])).to(DEVICE)
    mdl.load_state_dict(ckpt["state_dict"]); mdl.eval()
    id2lab = ckpt["id2label"]

    if lipidations is None:
        lipidations = ["none"] * len(sequences)
    texts = []
    for s, l in zip(sequences, lipidations):
        if ckpt["include_lipid"] and l and l.lower() not in ("none", "nan", ""):
            texts.append(f"{l[:3].upper()}-{s.upper()}")
        else:
            texts.append(s.upper())

    enc = tok(texts, padding=True, truncation=True,
              max_length=ckpt["max_len"], return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        _, logits = mdl(enc["input_ids"], enc["attention_mask"])
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    preds = probs.argmax(1)
    return pd.DataFrame({
        "sequence": sequences,
        "lipidation": lipidations,
        "prediction": [id2lab[i] for i in preds],
        **{f"prob_{id2lab[i]}": probs[:, i] for i in range(probs.shape[1])}
    })

# Example
demo = predict_sequences(["WWNWK", "NDRNK", "WWWDK"],
                         ["octanoyl", "octanoyl", "octanoyl"])
demo

In [ ]:
"""
ESM-2 Peptide Classifier - Model Evaluation Pipeline
======================================================
Loads a pre-trained ESM-2 checkpoint and evaluates performance on test/validation sets.
Generates ROC curves, confusion matrices, and detailed prediction CSVs.

Usage:
    python esm2_model_evaluation.py
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize


# ============================================================
# ============   USER-DEFINED PARAMETERS   ===================
# ============================================================
#Replace file path with actual location

INPUT_DIR        =  "data/database"
OUTPUT_DIR       =  "data/output"
TRAIN_FILE       = os.path.join(INPUT_DIR, "merged_output.xlsx")
VALID_FILE       = os.path.join(INPUT_DIR, "38-experimental-validation.xlsx")

BATCH_SIZE       = 32
SEED             = 42
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"
TEST_SIZE        = 0.2

torch.manual_seed(SEED); np.random.seed(SEED)
print("Device:", DEVICE)

# ============================================================
# ============   LOAD CHECKPOINT   ==========================
# ============================================================
model_path = os.path.join(OUTPUT_DIR, "esm2_classifier.pt")
checkpoint = torch.load(model_path, map_location=DEVICE)

# Extract saved parameters
state_dict = checkpoint["state_dict"]
label2id = checkpoint["label2id"]
id2label = checkpoint["id2label"]
ESM_MODEL_NAME = checkpoint["model_name"]
MAX_LEN = checkpoint["max_len"]
INCLUDE_LIPID = checkpoint["include_lipid"]

LABELS_TO_USE = sorted(label2id.keys())  # or use the original order
print("Loaded checkpoint:", model_path)
print("Labels:", LABELS_TO_USE)
print("Label map:", label2id)

# ============================================================
# ============   LOAD DATA (same as before)  ================
# ============================================================
def load_clean(path):
    df = pd.read_excel(path)
    df.columns = [c.strip().lower() for c in df.columns]
    needed = {"sequence", "category"}
    if not needed.issubset(df.columns):
        raise ValueError(f"{path} must contain columns: {needed}")
    if "lipidation" not in df.columns:
        df["lipidation"] = "none"
    df["sequence"]   = df["sequence"].astype(str).str.strip().str.upper()
    df["category"]   = df["category"].astype(str).str.strip()
    df["lipidation"] = df["lipidation"].astype(str).str.strip().str.lower().fillna("none")
    df = df[df["category"].isin(LABELS_TO_USE)].reset_index(drop=True)
    df = df.drop_duplicates(subset=["sequence", "lipidation"]).reset_index(drop=True)
    return df

train_df = load_clean(TRAIN_FILE)
print("Training data class distribution:")
print(train_df["category"].value_counts())

tokenizer = AutoTokenizer.from_pretrained(ESM_MODEL_NAME)

# ============================================================
# ============   RECREATE DATASET CLASS   ===================
# ============================================================
class PeptideDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=MAX_LEN, include_lipid=INCLUDE_LIPID):
        self.df = df.reset_index(drop=True)
        self.tok = tokenizer
        self.max_len = max_len
        self.include_lipid = include_lipid

    def __len__(self):
        return len(self.df)

    def _format_seq(self, row):
        seq = row["sequence"]
        if self.include_lipid and row["lipidation"] not in ("none", "nan", ""):
            return f"{row['lipidation'][:3].upper()}-{seq}"
        return seq

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = self._format_seq(row)
        enc = self.tok(text, truncation=True, padding="max_length",
                       max_length=self.max_len, return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(label2id[row["category"]], dtype=torch.long)
        return item

# ============================================================
# ============   RECREATE TRAIN/TEST SPLIT   ================
# ============================================================
tr_df, te_df = train_test_split(
    train_df, test_size=TEST_SIZE, random_state=SEED,
    stratify=train_df["category"]
)

test_ds  = PeptideDataset(te_df, tokenizer)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE)

print(f"Internal-test: {len(test_ds)}")

# ============================================================
# ============   REBUILD MODEL   ============================
# ============================================================
class ESM2Classifier(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.1):
        super().__init__()
        self.esm = AutoModel.from_pretrained(model_name)
        hidden = self.esm.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, num_labels)

    def forward(self, input_ids, attention_mask, labels=None):
        out = self.esm(input_ids=input_ids, attention_mask=attention_mask)
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1)
        logits = self.classifier(self.dropout(pooled))
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
        return loss, logits

# Initialize and load weights
model = ESM2Classifier(ESM_MODEL_NAME, num_labels=len(LABELS_TO_USE)).to(DEVICE)
model.load_state_dict(state_dict)
model.eval()
print(f"Model loaded and set to eval mode")

# ============================================================
# ============   EVALUATION FUNCTION   ======================
# ============================================================
def evaluate(loader):
    """Get predictions without labels needed"""
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            _, logits = model(batch["input_ids"], batch["attention_mask"])
            all_logits.append(logits.cpu().numpy())
            all_labels.append(batch["labels"].cpu().numpy())
    logits = np.concatenate(all_logits)
    labels = np.concatenate(all_labels)
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = probs.argmax(axis=1)
    return labels, preds, probs

def full_evaluation(loader, tag, df_for_export=None):
    """Full evaluation pipeline with macro-averaged ROC"""
    y_true, y_pred, probs = evaluate(loader)

    # ---- Metrics ----
    acc = accuracy_score(y_true, y_pred)
    print(f"\n=== {tag} ===")
    print(f"Accuracy: {acc:.4f}")
    print("\nClassification report:")
    print(classification_report(y_true, y_pred,
                                target_names=LABELS_TO_USE, digits=4, zero_division=0))

    # ---- Confusion matrix ----
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(LABELS_TO_USE))))
    plt.figure(figsize=(5.5, 4.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=LABELS_TO_USE, yticklabels=LABELS_TO_USE)
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"Confusion matrix — {tag}")
    plt.tight_layout()
    cm_path = os.path.join(OUTPUT_DIR, f"confusion_matrix_{tag}.png")
    plt.savefig(cm_path, dpi=200); plt.show()

    # ---- ROC / AUROC (one-vs-rest per class) ----
    y_bin = label_binarize(y_true, classes=list(range(len(LABELS_TO_USE))))
    auroc_per_class = {}
    
    plt.figure(figsize=(6, 5))
    for i, lab in enumerate(LABELS_TO_USE):
        if y_bin[:, i].sum() == 0:
            print(f"  [skip ROC for {lab}: no positives in {tag}]")
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
        a = auc(fpr, tpr)
        auroc_per_class[lab] = a
        plt.plot(fpr, tpr, label=f"{lab} (AUC={a:.3f})", alpha=0.7)
    
    plt.plot([0, 1], [0, 1], "k--", lw=1)
    plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
    plt.title(f"ROC — {tag} (per-class OvR)")
    plt.legend(loc="lower right"); plt.tight_layout()
    roc_path = os.path.join(OUTPUT_DIR, f"roc_{tag}.png")
    plt.savefig(roc_path, dpi=200); plt.show()

    # ---- MACRO-AVERAGED ROC CURVE ----
    print(f"\n--- Macro-averaged ROC ---")
    fpr_list = []
    tpr_list = []
    
    for i in range(len(LABELS_TO_USE)):
        if y_bin[:, i].sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
        fpr_list.append(fpr)
        tpr_list.append(tpr)
    
    # Interpolate all curves to common x-axis using NumPy (no scipy needed)
    mean_fpr = np.linspace(0, 1, 100)
    tpr_interp_list = []
    
    for fpr, tpr in zip(fpr_list, tpr_list):
        # Use numpy.interp instead of scipy
        tpr_interp = np.interp(mean_fpr, fpr, tpr)
        tpr_interp_list.append(tpr_interp)
    
    mean_tpr = np.mean(tpr_interp_list, axis=0)
    mean_tpr[0], mean_tpr[-1] = 0, 1  # Ensure endpoints
    macro_auc = auc(mean_fpr, mean_tpr)
    
    print(f"Macro-averaged AUROC ({tag}): {macro_auc:.4f}")
    
    # ---- Plot macro ROC ----
    plt.figure(figsize=(6, 5))
    plt.plot(mean_fpr, mean_tpr, label=f"Macro-avg (AUC={macro_auc:.3f})", lw=2.5, color="navy")
    plt.plot([0, 1], [0, 1], "k--", lw=1, label="Random classifier")
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title(f"Macro-averaged ROC — {tag}")
    plt.legend(loc="lower right")
    plt.tight_layout()
    macro_roc_path = os.path.join(OUTPUT_DIR, f"roc_macro_{tag}.png")
    plt.savefig(macro_roc_path, dpi=200); plt.show()
    print(f"Macro ROC saved: {macro_roc_path}")
    
    # ---- Save ROC curve as CSV ----
    roc_df = pd.DataFrame({
        "fpr": mean_fpr,
        "tpr": mean_tpr
    })
    roc_csv_path = os.path.join(OUTPUT_DIR, f"roc_macro_{tag}.csv")
    roc_df.to_csv(roc_csv_path, index=False)
    print(f"ROC curve (FPR/TPR) saved: {roc_csv_path}")
    
    # ---- Save ROC curve as CSV ----
    roc_df = pd.DataFrame({
        "fpr": mean_fpr,
        "tpr": mean_tpr
    })
    roc_csv_path = os.path.join(OUTPUT_DIR, f"roc_macro_{tag}.csv")
    roc_df.to_csv(roc_csv_path, index=False)
    print(f"ROC curve (FPR/TPR) saved: {roc_csv_path}")
    
    # ---- Macro AUROC (original) ----
    try:
        present = [i for i, lab in enumerate(LABELS_TO_USE) if y_bin[:, i].sum() > 0]
        if len(present) >= 2:
            macro_sklearn = roc_auc_score(y_bin[:, present], probs[:, present],
                                          average="macro", multi_class="ovr")
            print(f"Macro AUROC (sklearn, {tag}): {macro_sklearn:.4f}")
        else:
            macro_sklearn = float("nan")
    except Exception as e:
        macro_sklearn = float("nan")
        print("Macro AUROC (sklearn) could not be computed:", e)

    # ---- Save predictions to CSV ----
    if df_for_export is not None:
        out = df_for_export.copy().reset_index(drop=True)
        out["true_label"] = [id2label[i] for i in y_true]
        out["pred_label"] = [id2label[i] for i in y_pred]
        for i, lab in enumerate(LABELS_TO_USE):
            out[f"prob_{lab}"] = probs[:, i]
        out["correct"] = (out["true_label"] == out["pred_label"])
        csv_path = os.path.join(OUTPUT_DIR, f"predictions_{tag}.csv")
        out.to_csv(csv_path, index=False)
        print(f"Predictions saved to {csv_path}")

    return {
        "accuracy": acc, 
        "auroc_per_class": auroc_per_class,
        "macro_auroc": macro_auc,  # Using interpolated macro AUC
        "confusion_matrix": cm.tolist()
    }

# ============================================================
# ============   RUN EVALUATION   ============================
# ============================================================
internal_metrics = full_evaluation(test_loader, "internal_test", df_for_export=te_df)
print("\n✓ Evaluation complete!")
print("Metrics:", internal_metrics)

# ============================================================
# ============   RUN EVALUATION   ============================
# ============================================================
internal_metrics = full_evaluation(test_loader, "internal_test", df_for_export=te_df)
print("\n✓ Evaluation complete!")
print("Metrics:", internal_metrics)